# Capital vs Human Capital in Productivity Growth

**Original Question:** What drives productivity growth across countries — is it capital deepening or human capital accumulation?

_Exported from PlotStudio_

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go


# PlotStudio stub — no-op outside the app
def add_to_workspace(df):
    pass


# Load source data
df_pwt110 = pd.read_csv("data/df_pwt110.csv")
df_pwt110_panel_reg_fe = pd.read_csv("data/df_pwt110_panel_reg_fe.csv")

## Task 1: Build a focused country-year panel with productivity, capital per worker, and human capital growth measures, validate sample definition, and ensure stationarity assumptions are handled before any association analysis.

**Acceptance Criteria:** A cleaned and well-documented panel DataFrame with derived growth variables (Δlog TFP, Δlog K/L, Δlog hc), clear sample counts by country and year, and stationarity test results showing that differenced/log-differenced series are appropriate for subsequent regression and correlation analyses.

### 1.1: Subset df_pwt110 to core variables, define the working panel, and validate the unit of analysis and coverage.
_Output: print_

In [ ]:
import numpy as np
import pandas as pd

# Build core panel with country identifiers, year, key variables, and quality flags
core_cols = [
    "countrycode",
    "country",
    "year",
    "rtfpna",
    "ctfp",
    "rkna",
    "emp",
    "hc",
    "pop",
    "rgdpe",
    "i_outlier",
    "i_irr",
]
df_pwt110_panel_core = df_pwt110[core_cols].copy()

# Unified productivity level: prefer rtfpna, fall back to ctfp
if "rtfpna" in df_pwt110_panel_core.columns and "ctfp" in df_pwt110_panel_core.columns:
    df_pwt110_panel_core["tfp_level"] = df_pwt110_panel_core["rtfpna"].combine_first(
        df_pwt110_panel_core["ctfp"]
    )
else:
    df_pwt110_panel_core["tfp_level"] = np.nan

# Ensure year is numeric for sorting/reporting
if not pd.api.types.is_numeric_dtype(df_pwt110_panel_core["year"]):
    df_pwt110_panel_core["year"] = pd.to_numeric(
        df_pwt110_panel_core["year"], errors="coerce"
    )

# Coverage summary: complete observations on the four core variables
core_vars = ["tfp_level", "rkna", "emp", "hc"]
df_complete_obs = df_pwt110_panel_core.dropna(subset=core_vars).copy()

df_country_coverage = (
    df_complete_obs.groupby(["countrycode", "country"], dropna=False)
    .size()
    .reset_index(name="complete_years")
)

df_coverage_distribution = (
    df_country_coverage["complete_years"].value_counts().sort_index().reset_index()
)
df_coverage_distribution.columns = ["complete_years", "country_count"]

total_countries = df_country_coverage.shape[0]
num_15 = int((df_country_coverage["complete_years"] >= 15).sum())
num_20 = int((df_country_coverage["complete_years"] >= 20).sum())

print("Coverage summary for complete observations on tfp_level, rkna, emp, and hc:")
print(df_coverage_distribution)
print(f"Total countries with at least one complete observation: {total_countries}")
print(f"Countries with at least 15 complete years: {num_15}")
print(f"Countries with at least 20 complete years: {num_20}")

# Restrict to countries with at least 15 complete years, keeping all years for those countries
country_keep = df_country_coverage.loc[
    df_country_coverage["complete_years"] >= 15, "countrycode"
]
df_pwt110_panel_core = df_pwt110_panel_core[
    df_pwt110_panel_core["countrycode"].isin(country_keep)
].copy()

df_pwt110_panel_core = df_pwt110_panel_core.sort_values(
    ["countrycode", "year"]
).reset_index(drop=True)

# Save to workspace
add_to_workspace(df_pwt110_panel_core)

# Small sorted sample for a few countries
sample_countries = (
    df_pwt110_panel_core["countrycode"]
    .dropna()
    .sort_values()
    .drop_duplicates()
    .head(3)
    .tolist()
)
df_sample = (
    df_pwt110_panel_core[df_pwt110_panel_core["countrycode"].isin(sample_countries)]
    .sort_values(["countrycode", "year"])
    .head(15)
)
print("Sample of the restricted panel core:")
print(
    df_sample[
        [
            "countrycode",
            "country",
            "year",
            "tfp_level",
            "rkna",
            "emp",
            "hc",
            "pop",
            "rgdpe",
            "i_outlier",
            "i_irr",
        ]
    ]
)
print(
    "Note: countries are the independent units, while yearly observations are repeated measures, so panel methods will be needed later."
)


### 1.2: Construct capital-per-worker, GDP-per-capita, and growth-rate variables that will be used as productivity, capital deepening, and human capital accumulation measures.
_Output: print_

In [ ]:
import numpy as np
import pandas as pd

# Work on a copy so the existing workspace object is not overwritten in place
if not pd.api.types.is_numeric_dtype(df_pwt110_panel_core["year"]):
    df_pwt110_panel_core["year"] = pd.to_numeric(
        df_pwt110_panel_core["year"], errors="coerce"
    )

df_pwt110_panel_core_updated = df_pwt110_panel_core.sort_values(
    ["countrycode", "year"]
).copy()

# Level measures
with np.errstate(divide="ignore", invalid="ignore"):
    df_pwt110_panel_core_updated["capital_per_worker"] = (
        df_pwt110_panel_core_updated["rkna"] / df_pwt110_panel_core_updated["emp"]
    )
    df_pwt110_panel_core_updated["gdp_per_capita"] = (
        df_pwt110_panel_core_updated["rgdpe"] / df_pwt110_panel_core_updated["pop"]
    )

# Log versions with non-positive values treated as missing
for col_raw, col_log in [
    ("tfp_level", "log_tfp_level"),
    ("capital_per_worker", "log_capital_per_worker"),
    ("hc", "log_hc"),
    ("gdp_per_capita", "log_gdp_per_capita"),
]:
    df_pwt110_panel_core_updated[col_raw] = pd.to_numeric(
        df_pwt110_panel_core_updated[col_raw], errors="coerce"
    )
    df_pwt110_panel_core_updated[col_log] = np.where(
        df_pwt110_panel_core_updated[col_raw] > 0,
        np.log(df_pwt110_panel_core_updated[col_raw]),
        np.nan,
    )

# Keep panel intact except for dropping the first row of each country where growth is undefined
# Compute year-over-year log differences within country
for log_col, growth_col in [
    ("log_tfp_level", "growth_log_tfp_level"),
    ("log_capital_per_worker", "growth_log_capital_per_worker"),
    ("log_hc", "growth_log_hc"),
]:
    df_pwt110_panel_core_updated[growth_col] = df_pwt110_panel_core_updated.groupby(
        "countrycode"
    )[log_col].diff()

# Drop only the mechanically undefined first row per country based on available year order
country_first_row_mask = (
    df_pwt110_panel_core_updated.groupby("countrycode").cumcount() == 0
)
df_pwt110_panel_core_updated = (
    df_pwt110_panel_core_updated.loc[~country_first_row_mask]
    .copy()
    .reset_index(drop=True)
)

# Summary statistics for the transformed variables
summary_cols = [
    "capital_per_worker",
    "gdp_per_capita",
    "log_tfp_level",
    "log_capital_per_worker",
    "log_hc",
    "log_gdp_per_capita",
    "growth_log_tfp_level",
    "growth_log_capital_per_worker",
    "growth_log_hc",
]
df_summary_stats = df_pwt110_panel_core_updated[summary_cols].describe().T
print("Summary statistics for level, log, and growth measures:")
print(df_summary_stats)

# Non-missing shares for each growth series
growth_cols = [
    "growth_log_tfp_level",
    "growth_log_capital_per_worker",
    "growth_log_hc",
]
df_growth_shares = pd.DataFrame(
    {
        "growth_series": growth_cols,
        "non_missing_share": [
            df_pwt110_panel_core_updated[col].notna().mean() for col in growth_cols
        ],
        "non_missing_count": [
            df_pwt110_panel_core_updated[col].notna().sum() for col in growth_cols
        ],
    }
)
print("Non-missing shares for growth series:")
print(df_growth_shares)

# Count rows where all three growth series are jointly available
joint_mask = df_pwt110_panel_core_updated[growth_cols].notna().all(axis=1)
df_joint_growth = df_pwt110_panel_core_updated.loc[joint_mask].copy()
print(f"Rows with all three growth series jointly available: {len(df_joint_growth)}")

# Replace the workspace DataFrame with the updated version
# This keeps the required variable name while preserving the transformed panel
if "df_pwt110_panel_core" in globals():
    df_pwt110_panel_core = df_pwt110_panel_core_updated.copy()
else:
    df_pwt110_panel_core = df_pwt110_panel_core_updated.copy()

add_to_workspace(df_pwt110_panel_core)


### 1.3: Verify stationarity properties of TFP, capital per worker, and human capital series using ADF tests on levels and first differences before running correlations or regressions.
_Output: print_

In [ ]:
import numpy as np
import pandas as pd
from statsmodels.tsa.stattools import adfuller

# Work on the existing panel without recreating it
if not pd.api.types.is_numeric_dtype(df_pwt110_panel_core["year"]):
    df_pwt110_panel_core["year"] = pd.to_numeric(
        df_pwt110_panel_core["year"], errors="coerce"
    )

# Logged level series and their corresponding growth series
series_map = [
    ("log_tfp_level", "growth_log_tfp_level", "productivity"),
    ("log_capital_per_worker", "growth_log_capital_per_worker", "capital_per_worker"),
    ("log_hc", "growth_log_hc", "human_capital"),
]

# Count usable observations by country across the logged level series
country_span = (
    df_pwt110_panel_core.groupby(["countrycode", "country"], dropna=False)
    .agg(
        usable_log_tfp=("log_tfp_level", "count"),
        usable_log_capital=("log_capital_per_worker", "count"),
        usable_log_hc=("log_hc", "count"),
    )
    .reset_index()
)
country_span["usable_span_min"] = country_span[
    ["usable_log_tfp", "usable_log_capital", "usable_log_hc"]
].min(axis=1)

# Select up to 10 countries with the longest usable spans, keeping those with the best overall coverage
country_span_selected = country_span.sort_values(
    [
        "usable_span_min",
        "usable_log_tfp",
        "usable_log_capital",
        "usable_log_hc",
        "countrycode",
    ],
    ascending=[False, False, False, False, True],
).head(10)

selected_countries = country_span_selected["countrycode"].tolist()

# Helper to run ADF safely
adf_rows = []
for countrycode in selected_countries:
    country_name = country_span_selected.loc[
        country_span_selected["countrycode"] == countrycode, "country"
    ].iloc[0]
    df_country = (
        df_pwt110_panel_core[df_pwt110_panel_core["countrycode"] == countrycode]
        .sort_values("year")
        .copy()
    )

    for level_col, growth_col, var_label in series_map:
        for col_name, series_type in [(level_col, "level"), (growth_col, "difference")]:
            series = pd.to_numeric(df_country[col_name], errors="coerce").dropna()
            if len(series) < 8:
                adf_rows.append(
                    {
                        "country": country_name,
                        "countrycode": countrycode,
                        "variable": var_label,
                        "series_type": series_type,
                        "test_statistic": np.nan,
                        "p_value": np.nan,
                        "lag_length": np.nan,
                        "reject_unit_root_5pct": False,
                        "n_obs": len(series),
                    }
                )
                continue
            try:
                adf_result = adfuller(series, autolag="AIC")
                adf_rows.append(
                    {
                        "country": country_name,
                        "countrycode": countrycode,
                        "variable": var_label,
                        "series_type": series_type,
                        "test_statistic": adf_result[0],
                        "p_value": adf_result[1],
                        "lag_length": adf_result[2],
                        "reject_unit_root_5pct": bool(adf_result[1] < 0.05),
                        "n_obs": len(series),
                    }
                )
            except Exception:
                adf_rows.append(
                    {
                        "country": country_name,
                        "countrycode": countrycode,
                        "variable": var_label,
                        "series_type": series_type,
                        "test_statistic": np.nan,
                        "p_value": np.nan,
                        "lag_length": np.nan,
                        "reject_unit_root_5pct": False,
                        "n_obs": len(series),
                    }
                )


df_adf_results = pd.DataFrame(adf_rows)
df_adf_results = (
    df_adf_results[
        [
            "country",
            "countrycode",
            "variable",
            "series_type",
            "test_statistic",
            "p_value",
            "lag_length",
            "reject_unit_root_5pct",
            "n_obs",
        ]
    ]
    .sort_values(["country", "variable", "series_type"])
    .reset_index(drop=True)
)

# Compact summary: rejection counts in levels vs differences
summary_counts = (
    df_adf_results.groupby(["series_type", "reject_unit_root_5pct"])
    .size()
    .unstack(fill_value=0)
    .rename(columns={False: "do_not_reject", True: "reject"})
    .reset_index()
)
summary_counts["total_tests"] = (
    summary_counts["reject"] + summary_counts["do_not_reject"]
)
summary_counts["rejection_rate"] = (
    summary_counts["reject"] / summary_counts["total_tests"]
)

level_rejections = int(
    df_adf_results.loc[
        df_adf_results["series_type"] == "level", "reject_unit_root_5pct"
    ].sum()
)
diff_rejections = int(
    df_adf_results.loc[
        df_adf_results["series_type"] == "difference", "reject_unit_root_5pct"
    ].sum()
)
total_level_tests = int((df_adf_results["series_type"] == "level").sum())
total_diff_tests = int((df_adf_results["series_type"] == "difference").sum())

print("Selected countries with the longest usable spans across logged level series:")
print(
    country_span_selected[
        [
            "country",
            "countrycode",
            "usable_log_tfp",
            "usable_log_capital",
            "usable_log_hc",
            "usable_span_min",
        ]
    ]
)
print("ADFuller test results:")
print(df_adf_results)
print("Unit-root rejection summary by series type:")
print(summary_counts)
print(f"Level series: {level_rejections} rejections out of {total_level_tests} tests")
print(
    f"Difference series: {diff_rejections} rejections out of {total_diff_tests} tests"
)

if diff_rejections >= level_rejections:
    print(
        "Conclusion: the differenced/log-differenced series are more stationary overall, so subsequent analysis should rely on differences to reduce spurious correlation risk."
    )
else:
    print(
        "Conclusion: differences do not clearly dominate levels in stationarity tests, but differenced/log-differenced series remain the safer choice for reducing spurious correlation risk in panel regressions."
    )


### 1.4: Visualize the distribution of productivity growth across country-years to understand central tendency, spread, and outliers.
_Output: figure_

In [ ]:
import plotly.express as px

# Prepare non-missing productivity growth observations
if "df_pwt110_panel_core" not in globals():
    raise ValueError("df_pwt110_panel_core is required but not found in the namespace.")

df_productivity_growth = df_pwt110_panel_core[
    ["growth_log_tfp_level", "countrycode", "country", "year"]
].copy()
df_productivity_growth = df_productivity_growth.dropna(
    subset=["growth_log_tfp_level"]
).copy()

# Compute center and dispersion for the title
mean_growth = df_productivity_growth["growth_log_tfp_level"].mean()
std_growth = df_productivity_growth["growth_log_tfp_level"].std()

fig = px.histogram(
    df_productivity_growth,
    x="growth_log_tfp_level",
    nbins=40,
    title=f"Productivity Growth Is Centered Near {mean_growth:.3f} with Dispersion of {std_growth:.3f}",
    labels={
        "growth_log_tfp_level": "Productivity growth (log difference)",
        "count": "Country-year observations",
    },
)
fig.update_traces(marker_line_color="black", marker_line_width=1)
fig.update_layout(
    xaxis_title="Productivity growth (log difference)",
    yaxis_title="Country-year observations",
)
fig.show()


## Task 2: Quantify and visualize the simple associations between productivity growth and each driver—capital deepening and human capital accumulation—before moving to multivariate panel models.

**Acceptance Criteria:** Printed correlation and simple fixed-effects regression summaries showing the strength and sign of associations between dlog_tfp and each of dlog_k_per_worker and dlog_hc, plus clear scatter plots illustrating these relationships and any obvious nonlinearities or heterogeneity.

### 2.1: Define the core regression sample and derive income groups for heterogeneity analysis.
_Output: print_

In [ ]:
import pandas as pd

# Build regression sample from the existing panel core without recreating upstream data

df_pwt110_panel_reg = df_pwt110_panel_core[
    [
        "countrycode",
        "country",
        "year",
        "growth_log_tfp_level",
        "growth_log_capital_per_worker",
        "growth_log_hc",
        "gdp_per_capita",
    ]
].copy()

df_pwt110_panel_reg = df_pwt110_panel_reg.dropna(
    subset=["growth_log_tfp_level", "growth_log_capital_per_worker", "growth_log_hc"]
).copy()

# Country-level long-run average GDP per capita from the core panel
_df_country_income = (
    df_pwt110_panel_core[["countrycode", "country", "gdp_per_capita"]]
    .groupby(["countrycode", "country"], dropna=False)
    .agg(avg_gdp_per_capita=("gdp_per_capita", "mean"))
    .reset_index()
)

# Quartile assignment across countries with non-missing average GDP per capita
_df_income_valid = _df_country_income.dropna(subset=["avg_gdp_per_capita"]).copy()

try:
    _df_income_valid["income_quartile"] = pd.qcut(
        _df_income_valid["avg_gdp_per_capita"],
        q=4,
        labels=["Q1_lowest", "Q2", "Q3", "Q4_highest"],
        duplicates="drop",
    )
except ValueError:
    _df_income_valid["income_quartile"] = pd.cut(
        _df_income_valid["avg_gdp_per_capita"],
        bins=4,
        labels=["Q1_lowest", "Q2", "Q3", "Q4_highest"],
        include_lowest=True,
    )

# If qcut dropped bins due to ties, ensure labels are still valid and ordered
if _df_income_valid["income_quartile"].isna().any():
    _df_income_valid["income_quartile"] = pd.cut(
        _df_income_valid["avg_gdp_per_capita"],
        bins=4,
        labels=["Q1_lowest", "Q2", "Q3", "Q4_highest"],
        include_lowest=True,
    )

_df_income_labels = _df_income_valid[
    ["countrycode", "income_quartile", "avg_gdp_per_capita"]
].copy()

# Merge income group labels into the regression sample

df_pwt110_panel_reg = df_pwt110_panel_reg.merge(
    _df_income_labels, on="countrycode", how="left", validate="many_to_one"
).reset_index(drop=True)

# Clean up helper columns and ensure a tidy regression sample

df_pwt110_panel_reg = df_pwt110_panel_reg[
    [
        "countrycode",
        "country",
        "year",
        "growth_log_tfp_level",
        "growth_log_capital_per_worker",
        "growth_log_hc",
        "avg_gdp_per_capita",
        "income_quartile",
    ]
].copy()

# Print requested summaries
_total_obs = len(df_pwt110_panel_reg)
_unique_countries = df_pwt110_panel_reg["countrycode"].nunique()

print(f"Total observations in regression sample: {_total_obs}")
print(f"Unique countries in regression sample: {_unique_countries}")

_df_income_summary = (
    df_pwt110_panel_reg.groupby("income_quartile", dropna=False)
    .agg(observations=("countrycode", "size"), countries=("countrycode", "nunique"))
    .reset_index()
    .sort_values("income_quartile")
)

print("Observation and country counts by income group:")
print(_df_income_summary)

# Save to workspace
add_to_workspace(df_pwt110_panel_reg)


### 2.2: Estimate simple correlations between productivity growth and each driver to gauge raw association strength.
_Output: print_

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import pearsonr, spearmanr

# Use the existing regression sample directly
if "df_pwt110_panel_reg" not in globals():
    raise ValueError("df_pwt110_panel_reg is required but not found in the namespace.")

# Ensure the needed columns are numeric
for col in ["growth_log_tfp_level", "growth_log_capital_per_worker", "growth_log_hc"]:
    df_pwt110_panel_reg[col] = pd.to_numeric(df_pwt110_panel_reg[col], errors="coerce")


# Magnitude labels based on absolute correlation size
def _magnitude_label(r):
    ar = abs(r)
    if ar < 0.10:
        return "negligible"
    if ar < 0.30:
        return "small"
    if ar < 0.50:
        return "moderate"
    return "strong"


# Helper for pairwise correlations


def _corr_stats(df_in, y_col, x_col):
    df_pair = df_in[[y_col, x_col]].dropna().copy()
    n_obs = len(df_pair)
    if n_obs < 3:
        return {
            "pearson_r": np.nan,
            "pearson_p": np.nan,
            "spearman_r": np.nan,
            "spearman_p": np.nan,
            "n_obs": n_obs,
            "pearson_label": "insufficient",
            "spearman_label": "insufficient",
        }

    try:
        pearson_r, pearson_p = pearsonr(df_pair[y_col], df_pair[x_col])
    except Exception:
        pearson_r, pearson_p = np.nan, np.nan

    try:
        spearman_r, spearman_p = spearmanr(df_pair[y_col], df_pair[x_col])
    except Exception:
        spearman_r, spearman_p = np.nan, np.nan

    return {
        "pearson_r": pearson_r,
        "pearson_p": pearson_p,
        "spearman_r": spearman_r,
        "spearman_p": spearman_p,
        "n_obs": n_obs,
        "pearson_label": (
            _magnitude_label(pearson_r) if pd.notna(pearson_r) else "unknown"
        ),
        "spearman_label": (
            _magnitude_label(spearman_r) if pd.notna(spearman_r) else "unknown"
        ),
    }


# Overall correlations
pairs = [
    (
        "growth_log_tfp_level",
        "growth_log_capital_per_worker",
        "Productivity growth vs capital deepening",
    ),
    (
        "growth_log_tfp_level",
        "growth_log_hc",
        "Productivity growth vs human capital growth",
    ),
]

print("Overall correlations in df_pwt110_panel_reg:")
for y_col, x_col, label in pairs:
    stats_dict = _corr_stats(df_pwt110_panel_reg, y_col, x_col)
    print(label)
    print(
        f"Pearson: r={stats_dict['pearson_r']:.4f}, p={stats_dict['pearson_p']:.4g}, n={stats_dict['n_obs']}, magnitude={stats_dict['pearson_label']}"
    )
    print(
        f"Spearman: r={stats_dict['spearman_r']:.4f}, p={stats_dict['spearman_p']:.4g}, n={stats_dict['n_obs']}, magnitude={stats_dict['spearman_label']}"
    )

# Income-group correlations if adequate sample sizes are available
if "income_quartile" in df_pwt110_panel_reg.columns:
    df_group_summary = (
        df_pwt110_panel_reg.groupby("income_quartile", dropna=False)
        .agg(
            observations=("countrycode", "size"),
            countries=("countrycode", "nunique"),
            nonmissing_tfp=("growth_log_tfp_level", "count"),
            nonmissing_capital=("growth_log_capital_per_worker", "count"),
            nonmissing_hc=("growth_log_hc", "count"),
        )
        .reset_index()
    )

    print("Income group sample sizes:")
    print(df_group_summary)
    print(
        "Income-group correlations (reported only when the pairwise sample is adequate):"
    )

    for income_group in ["Q1_lowest", "Q2", "Q3", "Q4_highest"]:
        df_group = df_pwt110_panel_reg[
            df_pwt110_panel_reg["income_quartile"] == income_group
        ].copy()
        if df_group.empty:
            print(f"{income_group}: no observations")
            continue

        print(f"{income_group}:")
        for y_col, x_col, label in pairs:
            stats_dict = _corr_stats(df_group, y_col, x_col)
            if stats_dict["n_obs"] < 10:
                print(f"{label}: insufficient sample (n={stats_dict['n_obs']})")
            else:
                print(
                    f"{label} | Pearson r={stats_dict['pearson_r']:.4f}, p={stats_dict['pearson_p']:.4g}, n={stats_dict['n_obs']}, magnitude={stats_dict['pearson_label']}"
                )
                print(
                    f"{label} | Spearman r={stats_dict['spearman_r']:.4f}, p={stats_dict['spearman_p']:.4g}, n={stats_dict['n_obs']}, magnitude={stats_dict['spearman_label']}"
                )


### 2.3: Visualize the relationship between capital deepening and productivity growth across country-years and income groups.
_Output: figure_

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px

# Prepare plotting data from the existing regression sample

df_scatter_plot = df_pwt110_panel_reg[
    [
        "growth_log_capital_per_worker",
        "growth_log_tfp_level",
        "income_quartile",
        "countrycode",
        "country",
        "year",
    ]
].copy()

df_scatter_plot["growth_log_capital_per_worker"] = pd.to_numeric(
    df_scatter_plot["growth_log_capital_per_worker"], errors="coerce"
)
df_scatter_plot["growth_log_tfp_level"] = pd.to_numeric(
    df_scatter_plot["growth_log_tfp_level"], errors="coerce"
)
df_scatter_plot["income_quartile"] = df_scatter_plot["income_quartile"].astype(str)

df_scatter_plot = df_scatter_plot.dropna(
    subset=["growth_log_capital_per_worker", "growth_log_tfp_level", "income_quartile"]
).copy()

x_mean = df_scatter_plot["growth_log_capital_per_worker"].mean()
x_std = df_scatter_plot["growth_log_capital_per_worker"].std()
y_mean = df_scatter_plot["growth_log_tfp_level"].mean()
y_std = df_scatter_plot["growth_log_tfp_level"].std()

fig = px.scatter(
    df_scatter_plot,
    x="growth_log_capital_per_worker",
    y="growth_log_tfp_level",
    color="income_quartile",
    trendline="ols",
    title=f"Capital Deepening and Productivity Growth Show a Weak, Diffuse Relationship (x̄={x_mean:.3f}, σx={x_std:.3f}; ȳ={y_mean:.3f}, σy={y_std:.3f})",
    labels={
        "growth_log_capital_per_worker": "Capital deepening (log difference)",
        "growth_log_tfp_level": "Productivity growth (log difference)",
        "income_quartile": "Income group",
    },
)
fig.update_traces(marker=dict(size=7, opacity=0.7))
fig.show()


### 2.4: Visualize the relationship between human capital growth and productivity growth across country-years and income groups.
_Output: figure_

In [ ]:
import plotly.express as px
import pandas as pd

# Prepare plotting data from the existing regression sample

df_scatter_plot_hc = df_pwt110_panel_reg[
    [
        "growth_log_hc",
        "growth_log_tfp_level",
        "income_quartile",
        "countrycode",
        "country",
        "year",
    ]
].copy()

df_scatter_plot_hc["growth_log_hc"] = pd.to_numeric(
    df_scatter_plot_hc["growth_log_hc"], errors="coerce"
)
df_scatter_plot_hc["growth_log_tfp_level"] = pd.to_numeric(
    df_scatter_plot_hc["growth_log_tfp_level"], errors="coerce"
)
df_scatter_plot_hc["income_quartile"] = df_scatter_plot_hc["income_quartile"].astype(
    str
)

df_scatter_plot_hc = df_scatter_plot_hc.dropna(
    subset=["growth_log_hc", "growth_log_tfp_level", "income_quartile"]
).copy()

x_mean = df_scatter_plot_hc["growth_log_hc"].mean()
x_std = df_scatter_plot_hc["growth_log_hc"].std()
y_mean = df_scatter_plot_hc["growth_log_tfp_level"].mean()
y_std = df_scatter_plot_hc["growth_log_tfp_level"].std()

fig = px.scatter(
    df_scatter_plot_hc,
    x="growth_log_hc",
    y="growth_log_tfp_level",
    color="income_quartile",
    trendline="ols",
    title=f"Human Capital Growth and Productivity Growth Show a Weak, Diffuse Relationship (x̄={x_mean:.3f}, σx={x_std:.3f}; ȳ={y_mean:.3f}, σy={y_std:.3f})",
    labels={
        "growth_log_hc": "Human capital growth (log difference)",
        "growth_log_tfp_level": "Productivity growth (log difference)",
        "income_quartile": "Income group",
    },
)
fig.update_traces(marker=dict(size=7, opacity=0.7))
fig.show()


### 2.5: Estimate bivariate panel regressions of productivity growth on each driver with country fixed effects to control for time-invariant heterogeneity.
_Output: print_

In [ ]:
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf

# Use the existing regression sample directly
if "df_pwt110_panel_reg" not in globals():
    raise ValueError("df_pwt110_panel_reg is required but not found in the namespace.")

# Ensure numeric inputs
for col in ["growth_log_tfp_level", "growth_log_capital_per_worker", "growth_log_hc"]:
    df_pwt110_panel_reg[col] = pd.to_numeric(df_pwt110_panel_reg[col], errors="coerce")

# Build country fixed-effects samples

df_fe_capital = (
    df_pwt110_panel_reg[
        [
            "countrycode",
            "country",
            "year",
            "growth_log_tfp_level",
            "growth_log_capital_per_worker",
        ]
    ]
    .dropna()
    .copy()
)

df_fe_hc = (
    df_pwt110_panel_reg[
        ["countrycode", "country", "year", "growth_log_tfp_level", "growth_log_hc"]
    ]
    .dropna()
    .copy()
)

# Fit country fixed-effects regressions with clustered standard errors by country
model_fe_capital = smf.ols(
    "growth_log_tfp_level ~ growth_log_capital_per_worker + C(countrycode)",
    data=df_fe_capital,
).fit(cov_type="cluster", cov_kwds={"groups": df_fe_capital["countrycode"]})

model_fe_hc = smf.ols(
    "growth_log_tfp_level ~ growth_log_hc + C(countrycode)", data=df_fe_hc
).fit(cov_type="cluster", cov_kwds={"groups": df_fe_hc["countrycode"]})

# Helper to print compact model summary


def print_fe_summary(model, df_model, coef_name, label):
    coef = model.params[coef_name]
    se = model.bse[coef_name]
    t_stat = model.tvalues[coef_name]
    p_val = model.pvalues[coef_name]
    ci_low, ci_high = model.conf_int().loc[coef_name].tolist()
    r_squared = model.rsquared
    n_obs = int(model.nobs)
    n_countries = int(df_model["countrycode"].nunique())

    print(label)
    print(f"Coefficient ({coef_name}): {coef:.6f}")
    print(f"Clustered SE: {se:.6f}")
    print(f"t-statistic: {t_stat:.4f}")
    print(f"p-value: {p_val:.4g}")
    print(f"95% CI: [{ci_low:.6f}, {ci_high:.6f}]")
    print(f"R-squared: {r_squared:.4f}")
    print(f"Observations: {n_obs}")
    print(f"Countries: {n_countries}")


print_fe_summary(
    model_fe_capital,
    df_fe_capital,
    "growth_log_capital_per_worker",
    "Country fixed-effects regression: Productivity growth on capital deepening",
)
print_fe_summary(
    model_fe_hc,
    df_fe_hc,
    "growth_log_hc",
    "Country fixed-effects regression: Productivity growth on human capital growth",
)

# Short comparison of relative strength
cap_t = abs(model_fe_capital.tvalues["growth_log_capital_per_worker"])
hc_t = abs(model_fe_hc.tvalues["growth_log_hc"])
cap_coef = abs(model_fe_capital.params["growth_log_capital_per_worker"])
hc_coef = abs(model_fe_hc.params["growth_log_hc"])

print("Comparison:")
if cap_t > hc_t:
    print(
        "The capital-deepening fixed-effects relationship appears stronger by absolute t-statistic."
    )
elif hc_t > cap_t:
    print(
        "The human-capital fixed-effects relationship appears stronger by absolute t-statistic."
    )
else:
    print(
        "The two fixed-effects relationships appear equally strong by absolute t-statistic."
    )

print(
    f"Absolute coefficient size: capital deepening = {cap_coef:.6f}, human capital growth = {hc_coef:.6f}"
)
print(
    "Interpretation: compare both the coefficient magnitude and statistical significance, but the stronger bivariate fixed-effects link is the one with the larger absolute t-statistic and smaller p-value."
)


## Task 3: Jointly estimate the contributions of capital deepening and human capital accumulation to productivity growth in a panel framework, assess model robustness and diagnostics, and explore how their relative importance varies by income group and via interactions.

**Acceptance Criteria:** Fixed-effects and random-effects (mixed) panel regression results with proper clustered standard errors and diagnostics, clear comparison of capital and human capital coefficients and partial contributions, evidence on income-group heterogeneity and interaction effects, and documented model adequacy checks.

### 3.1: Estimate a fixed-effects panel regression of productivity growth on both capital deepening and human capital growth, optionally including common time effects.
_Output: print_

In [ ]:
import pandas as pd
import statsmodels.formula.api as smf

# Build the joint FE regression dataframe from the existing core panel
_df_income_labels_fe = (
    df_pwt110_panel_reg[["countrycode", "income_quartile"]].drop_duplicates().copy()
)

df_pwt110_panel_reg_fe = df_pwt110_panel_core[
    [
        "countrycode",
        "country",
        "year",
        "growth_log_tfp_level",
        "growth_log_capital_per_worker",
        "growth_log_hc",
        "i_outlier",
        "i_irr",
    ]
].copy()

df_pwt110_panel_reg_fe = df_pwt110_panel_reg_fe.merge(
    _df_income_labels_fe,
    on="countrycode",
    how="left",
    validate="many_to_one",
).reset_index(drop=True)

df_pwt110_panel_reg_fe = df_pwt110_panel_reg_fe[
    [
        "countrycode",
        "country",
        "year",
        "growth_log_tfp_level",
        "growth_log_capital_per_worker",
        "growth_log_hc",
        "i_outlier",
        "i_irr",
        "income_quartile",
    ]
].copy()

df_pwt110_panel_reg_fe = (
    df_pwt110_panel_reg_fe.dropna(
        subset=[
            "growth_log_tfp_level",
            "growth_log_capital_per_worker",
            "growth_log_hc",
        ]
    )
    .copy()
    .reset_index(drop=True)
)

add_to_workspace(df_pwt110_panel_reg_fe)

# Fit country FE and country+year FE models with clustered SEs by country

df_fe_base = df_pwt110_panel_reg_fe.dropna(
    subset=[
        "growth_log_tfp_level",
        "growth_log_capital_per_worker",
        "growth_log_hc",
        "countrycode",
    ]
).copy()

model_fe_base = smf.ols(
    "growth_log_tfp_level ~ growth_log_capital_per_worker + growth_log_hc + C(countrycode)",
    data=df_fe_base,
).fit(cov_type="cluster", cov_kwds={"groups": df_fe_base["countrycode"]})

_df_fe_year = df_fe_base.dropna(subset=["year"]).copy()
_df_fe_year["year"] = pd.to_numeric(_df_fe_year["year"], errors="coerce")
_df_fe_year = _df_fe_year.dropna(subset=["year"]).copy()
_df_fe_year["year"] = _df_fe_year["year"].astype(int)

model_fe_year = smf.ols(
    "growth_log_tfp_level ~ growth_log_capital_per_worker + growth_log_hc + C(countrycode) + C(year)",
    data=_df_fe_year,
).fit(cov_type="cluster", cov_kwds={"groups": _df_fe_year["countrycode"]})

# Compact comparison table for the two driver coefficients and fit metrics


def _extract_row(model, var_name):
    return {
        "coef": model.params.get(var_name, float("nan")),
        "se": model.bse.get(var_name, float("nan")),
        "t": model.tvalues.get(var_name, float("nan")),
        "p": model.pvalues.get(var_name, float("nan")),
        "ci_low": (
            model.conf_int().loc[var_name, 0]
            if var_name in model.params.index
            else float("nan")
        ),
        "ci_high": (
            model.conf_int().loc[var_name, 1]
            if var_name in model.params.index
            else float("nan")
        ),
    }


_base_cap = _extract_row(model_fe_base, "growth_log_capital_per_worker")
_base_hc = _extract_row(model_fe_base, "growth_log_hc")
_year_cap = _extract_row(model_fe_year, "growth_log_capital_per_worker")
_year_hc = _extract_row(model_fe_year, "growth_log_hc")

df_comparison = pd.DataFrame(
    [
        {
            "model": "Country FE",
            "term": "Capital deepening",
            **_base_cap,
            "r_squared": model_fe_base.rsquared,
            "adj_r_squared": model_fe_base.rsquared_adj,
            "n_obs": int(model_fe_base.nobs),
            "countries": int(df_fe_base["countrycode"].nunique()),
        },
        {
            "model": "Country FE",
            "term": "Human capital growth",
            **_base_hc,
            "r_squared": model_fe_base.rsquared,
            "adj_r_squared": model_fe_base.rsquared_adj,
            "n_obs": int(model_fe_base.nobs),
            "countries": int(df_fe_base["countrycode"].nunique()),
        },
        {
            "model": "Country + Year FE",
            "term": "Capital deepening",
            **_year_cap,
            "r_squared": model_fe_year.rsquared,
            "adj_r_squared": model_fe_year.rsquared_adj,
            "n_obs": int(model_fe_year.nobs),
            "countries": int(_df_fe_year["countrycode"].nunique()),
        },
        {
            "model": "Country + Year FE",
            "term": "Human capital growth",
            **_year_hc,
            "r_squared": model_fe_year.rsquared,
            "adj_r_squared": model_fe_year.rsquared_adj,
            "n_obs": int(model_fe_year.nobs),
            "countries": int(_df_fe_year["countrycode"].nunique()),
        },
    ]
)

df_comparison = df_comparison[
    [
        "model",
        "term",
        "coef",
        "se",
        "t",
        "p",
        "ci_low",
        "ci_high",
        "r_squared",
        "adj_r_squared",
        "n_obs",
        "countries",
    ]
].copy()

print(df_comparison)


### 3.2: Estimate a random-effects (mixed) model and perform a Hausman-type comparison to decide between FE and RE specifications.
_Output: print_

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy.stats import chi2

# Build a clean regression-ready dataframe from the existing core panel
_df_income_labels_retry = (
    df_pwt110_panel_reg[["countrycode", "income_quartile"]].drop_duplicates().copy()
)

df_pwt110_panel_reg_fe_retry = df_pwt110_panel_core[
    [
        "countrycode",
        "country",
        "year",
        "growth_log_tfp_level",
        "growth_log_capital_per_worker",
        "growth_log_hc",
    ]
].copy()

df_pwt110_panel_reg_fe_retry = df_pwt110_panel_reg_fe_retry.merge(
    _df_income_labels_retry,
    on="countrycode",
    how="left",
    validate="many_to_one",
).reset_index(drop=True)

df_pwt110_panel_reg_fe_retry = df_pwt110_panel_reg_fe_retry[
    [
        "countrycode",
        "country",
        "year",
        "growth_log_tfp_level",
        "growth_log_capital_per_worker",
        "growth_log_hc",
        "income_quartile",
    ]
].copy()

for col in [
    "growth_log_tfp_level",
    "growth_log_capital_per_worker",
    "growth_log_hc",
    "year",
]:
    df_pwt110_panel_reg_fe_retry[col] = pd.to_numeric(
        df_pwt110_panel_reg_fe_retry[col], errors="coerce"
    )

df_pwt110_panel_reg_fe_retry = (
    df_pwt110_panel_reg_fe_retry.dropna(
        subset=[
            "growth_log_tfp_level",
            "growth_log_capital_per_worker",
            "growth_log_hc",
        ]
    )
    .copy()
    .reset_index(drop=True)
)

add_to_workspace(df_pwt110_panel_reg_fe_retry)

# Prepare the common estimation sample
_df_re_sample = df_pwt110_panel_reg_fe_retry[
    [
        "countrycode",
        "year",
        "growth_log_tfp_level",
        "growth_log_capital_per_worker",
        "growth_log_hc",
    ]
].copy()
_df_re_sample["countrycode"] = _df_re_sample["countrycode"].astype(str)
_df_re_sample["year"] = pd.to_numeric(_df_re_sample["year"], errors="coerce")
_df_re_sample = _df_re_sample.dropna().copy()

# Pooled OLS residual variance (for idiosyncratic error estimate)
pooled_model_retry = smf.ols(
    "growth_log_tfp_level ~ growth_log_capital_per_worker + growth_log_hc",
    data=_df_re_sample,
).fit()

# Within estimator to estimate sigma_e^2 from demeaned residuals
_df_within = _df_re_sample.copy()
for col in ["growth_log_tfp_level", "growth_log_capital_per_worker", "growth_log_hc"]:
    _df_within[f"{col}_dm"] = _df_within[col] - _df_within.groupby("countrycode")[
        col
    ].transform("mean")

within_model_retry = sm.OLS(
    _df_within["growth_log_tfp_level_dm"],
    sm.add_constant(
        _df_within[["growth_log_capital_per_worker_dm", "growth_log_hc_dm"]]
    ),
).fit()

sigma_e2 = float(np.mean(within_model_retry.resid**2))

# Variance components from pooled and within residuals
_df_resid_means = (
    pd.DataFrame(
        {
            "countrycode": _df_re_sample["countrycode"],
            "pooled_resid": pooled_model_retry.resid,
        }
    )
    .groupby("countrycode", as_index=False)
    .agg(
        resid_mean=("pooled_resid", "mean"),
        n_i=("pooled_resid", "size"),
    )
)

resid_mean_var = (
    float(_df_resid_means["resid_mean"].var(ddof=1))
    if len(_df_resid_means) > 1
    else 0.0
)
avg_t = float(_df_resid_means["n_i"].mean()) if len(_df_resid_means) > 0 else 1.0
sigma_u2 = max(0.0, resid_mean_var - sigma_e2 / max(avg_t, 1.0))

theta_retry = (
    1.0 - np.sqrt(sigma_e2 / (sigma_e2 + avg_t * sigma_u2))
    if (sigma_e2 + avg_t * sigma_u2) > 0
    else 0.0
)
theta_retry = float(np.clip(theta_retry, 0.0, 1.0))

# Quasi-demeaned FGLS random-effects estimator
_df_qd = _df_re_sample.merge(
    _df_resid_means[["countrycode", "n_i"]],
    on="countrycode",
    how="left",
    validate="many_to_one",
).copy()
for col in ["growth_log_tfp_level", "growth_log_capital_per_worker", "growth_log_hc"]:
    group_mean = _df_qd.groupby("countrycode")[col].transform("mean")
    _df_qd[f"{col}_qd"] = _df_qd[col] - theta_retry * group_mean

_df_qd["const_qd"] = 1.0 - theta_retry
X_qd = _df_qd[
    ["const_qd", "growth_log_capital_per_worker_qd", "growth_log_hc_qd"]
].astype(float)
y_qd = _df_qd["growth_log_tfp_level_qd"].astype(float)
re_model_retry = sm.OLS(y_qd, X_qd).fit(cov_type="HC3")

# Comparable country fixed-effects model on the same sample
fe_model_retry = smf.ols(
    "growth_log_tfp_level ~ growth_log_capital_per_worker + growth_log_hc + C(countrycode)",
    data=_df_re_sample,
).fit(cov_type="cluster", cov_kwds={"groups": _df_re_sample["countrycode"]})

# Align coefficients and covariance matrices for Hausman test
coef_names_retry = ["growth_log_capital_per_worker", "growth_log_hc"]
b_fe_retry = fe_model_retry.params[coef_names_retry].astype(float).values
v_fe_retry = (
    fe_model_retry.cov_params()
    .loc[coef_names_retry, coef_names_retry]
    .astype(float)
    .values
)

b_re_retry = np.array(
    [
        re_model_retry.params["growth_log_capital_per_worker_qd"],
        re_model_retry.params["growth_log_hc_qd"],
    ],
    dtype=float,
)
v_re_retry = (
    re_model_retry.cov_params()
    .loc[
        ["growth_log_capital_per_worker_qd", "growth_log_hc_qd"],
        ["growth_log_capital_per_worker_qd", "growth_log_hc_qd"],
    ]
    .astype(float)
    .values
)

b_diff_retry = b_fe_retry - b_re_retry
v_diff_retry = v_fe_retry - v_re_retry
v_diff_inv_retry = np.linalg.pinv(v_diff_retry)
hausman_stat_retry = float(b_diff_retry.T @ v_diff_inv_retry @ b_diff_retry)
hausman_df_retry = len(coef_names_retry)
hausman_p_retry = float(1 - chi2.cdf(hausman_stat_retry, df=hausman_df_retry))

# Compact comparison table


def _ci_series(model, term):
    if term in model.params.index:
        ci = model.conf_int().loc[term]
        return float(ci.iloc[0]), float(ci.iloc[1])
    return np.nan, np.nan


fe_cap_ci_low, fe_cap_ci_high = _ci_series(
    fe_model_retry, "growth_log_capital_per_worker"
)
fe_hc_ci_low, fe_hc_ci_high = _ci_series(fe_model_retry, "growth_log_hc")
re_cap_ci_low, re_cap_ci_high = _ci_series(
    re_model_retry, "growth_log_capital_per_worker_qd"
)
re_hc_ci_low, re_hc_ci_high = _ci_series(re_model_retry, "growth_log_hc_qd")

df_re_fe_comparison_retry = pd.DataFrame(
    [
        {
            "model": "FE",
            "term": "Capital deepening",
            "coef": float(fe_model_retry.params["growth_log_capital_per_worker"]),
            "se": float(fe_model_retry.bse["growth_log_capital_per_worker"]),
            "ci_low": fe_cap_ci_low,
            "ci_high": fe_cap_ci_high,
        },
        {
            "model": "FE",
            "term": "Human capital growth",
            "coef": float(fe_model_retry.params["growth_log_hc"]),
            "se": float(fe_model_retry.bse["growth_log_hc"]),
            "ci_low": fe_hc_ci_low,
            "ci_high": fe_hc_ci_high,
        },
        {
            "model": "RE",
            "term": "Capital deepening",
            "coef": float(re_model_retry.params["growth_log_capital_per_worker_qd"]),
            "se": float(re_model_retry.bse["growth_log_capital_per_worker_qd"]),
            "ci_low": re_cap_ci_low,
            "ci_high": re_cap_ci_high,
        },
        {
            "model": "RE",
            "term": "Human capital growth",
            "coef": float(re_model_retry.params["growth_log_hc_qd"]),
            "se": float(re_model_retry.bse["growth_log_hc_qd"]),
            "ci_low": re_hc_ci_low,
            "ci_high": re_hc_ci_high,
        },
    ]
)

df_re_fe_comparison_retry["theta"] = np.nan
df_re_fe_comparison_retry.loc[df_re_fe_comparison_retry["model"] == "RE", "theta"] = (
    theta_retry
)

df_re_fe_comparison_retry = df_re_fe_comparison_retry[
    ["model", "term", "coef", "se", "ci_low", "ci_high", "theta"]
].copy()

print("Random-effects via quasi-demeaned feasible GLS and comparable fixed effects:")
print(df_re_fe_comparison_retry)
print(f"Estimated theta used in quasi-demeaning: {theta_retry:.4f}")
print(f"Hausman statistic: {hausman_stat_retry:.4f}")
print(f"Hausman p-value: {hausman_p_retry:.4f}")

if pd.notna(hausman_p_retry) and hausman_p_retry < 0.05:
    print(
        "FE is preferred because the Hausman test rejects the random-effects specification."
    )
else:
    print("RE is preferred because the Hausman test does not reject random effects.")


### 3.3: Assess the adequacy of the chosen regression specification through heteroskedasticity, serial-correlation, functional-form, and multicollinearity diagnostics.
_Output: print_

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.diagnostic import (
    het_breuschpagan,
    het_white,
    acorr_breusch_godfrey,
    linear_reset,
)
from statsmodels.stats.outliers_influence import variance_inflation_factor

# Use the existing regression-ready panel and benchmark fixed-effects model
if "df_pwt110_panel_reg_fe" not in globals():
    raise ValueError(
        "df_pwt110_panel_reg_fe is required but not found in the namespace."
    )
if "model_fe_base" not in globals():
    raise ValueError("model_fe_base is required but not found in the namespace.")

# Rebuild a clean sample aligned to the benchmark model to ensure diagnostics are valid

df_diag_base = df_pwt110_panel_reg_fe[
    [
        "countrycode",
        "country",
        "year",
        "growth_log_tfp_level",
        "growth_log_capital_per_worker",
        "growth_log_hc",
    ]
].copy()

for col in [
    "growth_log_tfp_level",
    "growth_log_capital_per_worker",
    "growth_log_hc",
    "year",
]:
    df_diag_base[col] = pd.to_numeric(df_diag_base[col], errors="coerce")

df_diag_base = df_diag_base.dropna().copy()
df_diag_base["countrycode"] = df_diag_base["countrycode"].astype(str)
df_diag_base["year"] = df_diag_base["year"].astype(int)

# Fit a reduced benchmark FE model on the same estimation sample for diagnostics
model_diag = smf.ols(
    "growth_log_tfp_level ~ growth_log_capital_per_worker + growth_log_hc + C(countrycode)",
    data=df_diag_base,
).fit(cov_type="cluster", cov_kwds={"groups": df_diag_base["countrycode"]})

# Heteroskedasticity tests
X_aux = sm.add_constant(
    pd.DataFrame(
        {
            "growth_log_capital_per_worker": df_diag_base[
                "growth_log_capital_per_worker"
            ],
            "growth_log_hc": df_diag_base["growth_log_hc"],
        }
    )
)
bp_stat, bp_p, bp_f_stat, bp_f_p = het_breuschpagan(model_diag.resid, X_aux)
white_stat, white_p, white_f_stat, white_f_p = het_white(model_diag.resid, X_aux)

# Serial correlation test (Breusch-Godfrey)
# Use a low lag order suitable for annual panel data; the model is already estimated on pooled FE residuals.
bg_stat, bg_p, bg_f_stat, bg_f_p = acorr_breusch_godfrey(model_diag, nlags=2)

# Ramsey RESET for functional form
reset_result = linear_reset(model_diag, power=3, use_f=True)

# VIFs for the two main regressors in a reduced design matrix
X_vif = sm.add_constant(
    df_diag_base[["growth_log_capital_per_worker", "growth_log_hc"]].astype(float)
)
df_vif_summary = pd.DataFrame(
    {
        "regressor": ["growth_log_capital_per_worker", "growth_log_hc"],
        "vif": [
            variance_inflation_factor(X_vif.values, 1),
            variance_inflation_factor(X_vif.values, 2),
        ],
    }
)

# Tidy diagnostic summary

df_diag_summary = pd.DataFrame(
    [
        {
            "test": "Breusch-Pagan",
            "statistic": bp_stat,
            "p_value": bp_p,
        },
        {
            "test": "White",
            "statistic": white_stat,
            "p_value": white_p,
        },
        {
            "test": "Breusch-Godfrey (lag 2)",
            "statistic": bg_stat,
            "p_value": bg_p,
        },
        {
            "test": "Ramsey RESET",
            "statistic": reset_result.fvalue,
            "p_value": reset_result.pvalue,
        },
    ]
)

print("Diagnostic tests for the country fixed-effects benchmark model:")
print(df_diag_summary)
print("VIFs for the two main regressors:")
print(df_vif_summary)

# Plain-language conclusion
heteroskedasticity_flag = bool((bp_p < 0.05) or (white_p < 0.05))
autocorr_flag = bool(bg_p < 0.05)
reset_flag = bool(reset_result.pvalue < 0.05)
max_vif = float(df_vif_summary["vif"].max())

print("Conclusion:")
if heteroskedasticity_flag or autocorr_flag:
    print(
        "Clustered standard errors look justified because the diagnostics indicate at least one form of non-constant or serially correlated errors."
    )
else:
    print(
        "Clustered standard errors are not strongly demanded by these diagnostics, but they remain a safe choice in panel data."
    )

if reset_flag:
    print(
        "There is some evidence of functional-form misspecification from the RESET test, so the model may be missing relevant nonlinearities or interactions."
    )
else:
    print(
        "The RESET test does not strongly suggest misspecification, so the linear FE specification looks broadly acceptable."
    )

if max_vif >= 10:
    print(
        "Collinearity looks problematic: at least one VIF is high enough to warrant caution."
    )
else:
    print(
        "Collinearity between capital deepening and human capital growth does not look problematic; the VIFs are comfortably below common concern thresholds."
    )


### 3.4: Investigate whether the relative importance of capital deepening vs human capital growth varies by income group and whether there are interaction effects between the two drivers.
_Output: print_

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf

if "df_pwt110_panel_reg_fe" not in globals():
    raise ValueError(
        "df_pwt110_panel_reg_fe is required but not found in the namespace."
    )

# Clean estimation sample

df_interaction_sample = df_pwt110_panel_reg_fe[
    [
        "countrycode",
        "country",
        "year",
        "growth_log_tfp_level",
        "growth_log_capital_per_worker",
        "growth_log_hc",
        "income_quartile",
    ]
].copy()

for col in [
    "growth_log_tfp_level",
    "growth_log_capital_per_worker",
    "growth_log_hc",
    "year",
]:
    df_interaction_sample[col] = pd.to_numeric(
        df_interaction_sample[col], errors="coerce"
    )

df_interaction_sample["income_quartile"] = df_interaction_sample[
    "income_quartile"
].astype("object")
df_interaction_sample = df_interaction_sample.dropna(
    subset=[
        "growth_log_tfp_level",
        "growth_log_capital_per_worker",
        "growth_log_hc",
        "income_quartile",
        "countrycode",
    ]
).copy()

# Model with interactions between each driver and income group, plus country fixed effects
formula_income = (
    "growth_log_tfp_level ~ growth_log_capital_per_worker * C(income_quartile) "
    "+ growth_log_hc * C(income_quartile) + C(countrycode)"
)

model_income_interactions = smf.ols(formula_income, data=df_interaction_sample).fit(
    cov_type="cluster", cov_kwds={"groups": df_interaction_sample["countrycode"]}
)

# Recover group-specific effects relative to the reference group (the first category in alphabetical/order used by patsy)
cat_series = pd.Categorical(df_interaction_sample["income_quartile"])
base_group = cat_series.categories[0]
other_groups = list(cat_series.categories[1:])


def _safe_get(model, term):
    if term in model.params.index:
        return (
            float(model.params[term]),
            float(model.bse[term]),
            float(model.pvalues[term]),
        )
    return 0.0, 0.0, 1.0


cap_base_coef, cap_base_se, cap_base_p = _safe_get(
    model_income_interactions, "growth_log_capital_per_worker"
)
hc_base_coef, hc_base_se, hc_base_p = _safe_get(
    model_income_interactions, "growth_log_hc"
)

rows = []
for grp in cat_series.categories:
    if grp == base_group:
        cap_coef = cap_base_coef
        cap_se = cap_base_se
        cap_p = cap_base_p
        hc_coef = hc_base_coef
        hc_se = hc_base_se
        hc_p = hc_base_p
    else:
        cap_term = f"growth_log_capital_per_worker:C(income_quartile)[T.{grp}]"
        hc_term = f"growth_log_hc:C(income_quartile)[T.{grp}]"
        cap_int_coef, cap_int_se, _ = _safe_get(model_income_interactions, cap_term)
        hc_int_coef, hc_int_se, _ = _safe_get(model_income_interactions, hc_term)

        cap_coef = cap_base_coef + cap_int_coef
        hc_coef = hc_base_coef + hc_int_coef

        var_cap = 0.0
        if (
            "growth_log_capital_per_worker"
            in model_income_interactions.cov_params().index
            and cap_term in model_income_interactions.cov_params().index
        ):
            var_cap = (
                model_income_interactions.cov_params().loc[
                    "growth_log_capital_per_worker", "growth_log_capital_per_worker"
                ]
                + model_income_interactions.cov_params().loc[cap_term, cap_term]
                + 2.0
                * model_income_interactions.cov_params().loc[
                    "growth_log_capital_per_worker", cap_term
                ]
            )
        else:
            var_cap = cap_se**2 + cap_int_se**2

        var_hc = 0.0
        if (
            "growth_log_hc" in model_income_interactions.cov_params().index
            and hc_term in model_income_interactions.cov_params().index
        ):
            var_hc = (
                model_income_interactions.cov_params().loc[
                    "growth_log_hc", "growth_log_hc"
                ]
                + model_income_interactions.cov_params().loc[hc_term, hc_term]
                + 2.0
                * model_income_interactions.cov_params().loc["growth_log_hc", hc_term]
            )
        else:
            var_hc = hc_se**2 + hc_int_se**2

        cap_se = float(np.sqrt(max(var_cap, 0.0)))
        hc_se = float(np.sqrt(max(var_hc, 0.0)))
        cap_p = (
            float(
                2.0
                * (
                    1.0
                    - smf.ols(
                        "growth_log_tfp_level ~ 1", data=df_interaction_sample.head(2)
                    )
                    .fit()
                    .model.data.const_idx
                )
            )
            if False
            else np.nan
        )
        hc_p = np.nan

    rows.append(
        {
            "income_quartile": grp,
            "capital_deepening_effect": cap_coef,
            "capital_deepening_se": cap_se,
            "capital_deepening_sig": bool(
                pd.notna(cap_se) and cap_se > 0 and abs(cap_coef / cap_se) > 1.96
            ),
            "human_capital_effect": hc_coef,
            "human_capital_se": hc_se,
            "human_capital_sig": bool(
                pd.notna(hc_se) and hc_se > 0 and abs(hc_coef / hc_se) > 1.96
            ),
        }
    )

df_income_effects = pd.DataFrame(rows)

print(f"Income-group interaction model: baseline income group = {base_group}")
print(df_income_effects)

# Explicit interaction between capital deepening and human capital growth
model_driver_interaction = smf.ols(
    "growth_log_tfp_level ~ growth_log_capital_per_worker * growth_log_hc + C(countrycode)",
    data=df_interaction_sample,
).fit(cov_type="cluster", cov_kwds={"groups": df_interaction_sample["countrycode"]})

interaction_term = "growth_log_capital_per_worker:growth_log_hc"
interaction_coef = float(model_driver_interaction.params.get(interaction_term, np.nan))
interaction_se = float(model_driver_interaction.bse.get(interaction_term, np.nan))
interaction_p = float(model_driver_interaction.pvalues.get(interaction_term, np.nan))

print("Driver interaction model: capital deepening x human capital growth")
print(f"Coefficient: {interaction_coef:.6f}")
print(f"Standard error: {interaction_se:.6f}")
print(f"p-value: {interaction_p:.4g}")

if pd.notna(interaction_p) and interaction_p < 0.05:
    if interaction_coef > 0:
        interpretation = "The positive and statistically significant interaction suggests capital deepening and human capital growth reinforce each other in raising productivity growth."
    else:
        interpretation = "The negative and statistically significant interaction suggests capital deepening and human capital growth partially substitute for each other in explaining productivity growth."
else:
    interpretation = "The interaction is not statistically different from zero, so there is no strong evidence that the effect of one driver depends on the level of the other."

print(interpretation)


### 3.5: Assess robustness of key panel regression results to excluding observations flagged as outliers or irregular.
_Output: print_

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf

if "df_pwt110_panel_reg_fe" not in globals():
    raise ValueError(
        "df_pwt110_panel_reg_fe is required but not found in the namespace."
    )

# Build a clean sample by removing rows flagged as Outlier in either flag column, while keeping regular and missing flags.
df_pwt110_panel_reg_clean = df_pwt110_panel_reg_fe.copy()
df_pwt110_panel_reg_clean = (
    df_pwt110_panel_reg_clean[
        ~(
            df_pwt110_panel_reg_clean["i_outlier"].astype("object").eq("Outlier")
            | df_pwt110_panel_reg_clean["i_irr"].astype("object").eq("Outlier")
        )
    ]
    .copy()
    .reset_index(drop=True)
)

add_to_workspace(df_pwt110_panel_reg_clean)

# Ensure numeric inputs are clean for estimation
for col in ["growth_log_tfp_level", "growth_log_capital_per_worker", "growth_log_hc"]:
    df_pwt110_panel_reg_fe[col] = pd.to_numeric(
        df_pwt110_panel_reg_fe[col], errors="coerce"
    )
    df_pwt110_panel_reg_clean[col] = pd.to_numeric(
        df_pwt110_panel_reg_clean[col], errors="coerce"
    )

# Align samples for the same joint country FE model without year effects
full_cols = [
    "countrycode",
    "growth_log_tfp_level",
    "growth_log_capital_per_worker",
    "growth_log_hc",
]

_df_full_model = df_pwt110_panel_reg_fe[full_cols].dropna().copy()
_df_clean_model = df_pwt110_panel_reg_clean[full_cols].dropna().copy()

model_full = smf.ols(
    "growth_log_tfp_level ~ growth_log_capital_per_worker + growth_log_hc + C(countrycode)",
    data=_df_full_model,
).fit(cov_type="cluster", cov_kwds={"groups": _df_full_model["countrycode"]})

model_clean = smf.ols(
    "growth_log_tfp_level ~ growth_log_capital_per_worker + growth_log_hc + C(countrycode)",
    data=_df_clean_model,
).fit(cov_type="cluster", cov_kwds={"groups": _df_clean_model["countrycode"]})

terms = ["growth_log_capital_per_worker", "growth_log_hc"]
term_labels = {
    "growth_log_capital_per_worker": "Capital deepening",
    "growth_log_hc": "Human capital growth",
}

rows = []
for term in terms:
    full_coef = float(model_full.params.get(term, np.nan))
    full_ci = (
        model_full.conf_int().loc[term].tolist()
        if term in model_full.params.index
        else [np.nan, np.nan]
    )
    clean_coef = float(model_clean.params.get(term, np.nan))
    clean_ci = (
        model_clean.conf_int().loc[term].tolist()
        if term in model_clean.params.index
        else [np.nan, np.nan]
    )

    full_p = float(model_full.pvalues.get(term, np.nan))
    clean_p = float(model_clean.pvalues.get(term, np.nan))
    full_sig = pd.notna(full_p) and full_p < 0.05
    clean_sig = pd.notna(clean_p) and clean_p < 0.05

    sign_change = (
        pd.notna(full_coef)
        and pd.notna(clean_coef)
        and np.sign(full_coef) != np.sign(clean_coef)
    )
    magnitude_change = (
        pd.notna(full_coef)
        and pd.notna(clean_coef)
        and abs(full_coef - clean_coef) > max(0.05 * max(abs(full_coef), 1e-12), 1e-6)
    )
    sig_change = full_sig != clean_sig

    note_parts = []
    if sign_change:
        note_parts.append("sign changes")
    if magnitude_change:
        note_parts.append("magnitude changes materially")
    if sig_change:
        note_parts.append("significance changes")
    if not note_parts:
        note_parts.append("no material change")

    rows.append(
        {
            "term": term_labels[term],
            "full_coef": full_coef,
            "full_ci_low": full_ci[0],
            "full_ci_high": full_ci[1],
            "clean_coef": clean_coef,
            "clean_ci_low": clean_ci[0],
            "clean_ci_high": clean_ci[1],
            "note": ", ".join(note_parts),
            "full_p": full_p,
            "clean_p": clean_p,
        }
    )

df_comparison_clean = pd.DataFrame(rows)

df_comparison_clean = df_comparison_clean[
    [
        "term",
        "full_coef",
        "full_ci_low",
        "full_ci_high",
        "clean_coef",
        "clean_ci_low",
        "clean_ci_high",
        "note",
        "full_p",
        "clean_p",
    ]
].copy()

print(f"Full sample observations used: {int(model_full.nobs)}")
print(f"Clean sample observations used: {int(model_clean.nobs)}")
print("Compact comparison of joint country FE models without year effects:")
print(df_comparison_clean)


### 3.6: Probe why one driver appears more influential than the other by examining distributional and coverage differences across variables and income groups.
_Output: print_

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import pearsonr

# Use the existing regression-ready panel with complete growth measures
if "df_pwt110_panel_reg_fe" not in globals():
    raise ValueError(
        "df_pwt110_panel_reg_fe is required but not found in the namespace."
    )

# Ensure numeric types for the two driver variables
for col in ["growth_log_capital_per_worker", "growth_log_hc"]:
    df_pwt110_panel_reg_fe[col] = pd.to_numeric(
        df_pwt110_panel_reg_fe[col], errors="coerce"
    )

# Keep only rows with both drivers and income group available for the descriptive tables

df_driver_summary_base = df_pwt110_panel_reg_fe[
    ["growth_log_capital_per_worker", "growth_log_hc", "income_quartile"]
].copy()
df_driver_summary_base = df_driver_summary_base.dropna(
    subset=["growth_log_capital_per_worker", "growth_log_hc", "income_quartile"]
).copy()
df_driver_summary_base["income_quartile"] = df_driver_summary_base[
    "income_quartile"
].astype(str)

# Overall summary statistics
overall_stats = pd.DataFrame(
    {
        "variable": ["growth_log_capital_per_worker", "growth_log_hc"],
        "mean": [
            df_driver_summary_base["growth_log_capital_per_worker"].mean(),
            df_driver_summary_base["growth_log_hc"].mean(),
        ],
        "std": [
            df_driver_summary_base["growth_log_capital_per_worker"].std(),
            df_driver_summary_base["growth_log_hc"].std(),
        ],
        "min": [
            df_driver_summary_base["growth_log_capital_per_worker"].min(),
            df_driver_summary_base["growth_log_hc"].min(),
        ],
        "max": [
            df_driver_summary_base["growth_log_capital_per_worker"].max(),
            df_driver_summary_base["growth_log_hc"].max(),
        ],
        "n_obs": [
            df_driver_summary_base["growth_log_capital_per_worker"].notna().sum(),
            df_driver_summary_base["growth_log_hc"].notna().sum(),
        ],
    }
)

print("Overall summary statistics for the two drivers:")
print(overall_stats)

# By income group summary statistics

df_group_stats = (
    df_driver_summary_base.groupby("income_quartile", dropna=False)
    .agg(
        capital_mean=("growth_log_capital_per_worker", "mean"),
        capital_std=("growth_log_capital_per_worker", "std"),
        capital_min=("growth_log_capital_per_worker", "min"),
        capital_max=("growth_log_capital_per_worker", "max"),
        capital_n=("growth_log_capital_per_worker", "count"),
        hc_mean=("growth_log_hc", "mean"),
        hc_std=("growth_log_hc", "std"),
        hc_min=("growth_log_hc", "min"),
        hc_max=("growth_log_hc", "max"),
        hc_n=("growth_log_hc", "count"),
    )
    .reset_index()
    .sort_values("income_quartile")
)

print("Summary statistics by income group:")
print(df_group_stats)

# Correlations overall and by income group


def _safe_corr(df_in, x_col, y_col):
    df_pair = df_in[[x_col, y_col]].dropna().copy()
    n_obs = len(df_pair)
    if n_obs < 3:
        return np.nan, np.nan, n_obs
    try:
        r_val, p_val = pearsonr(df_pair[x_col], df_pair[y_col])
    except Exception:
        r_val, p_val = np.nan, np.nan
    return r_val, p_val, n_obs


overall_r, overall_p, overall_n = _safe_corr(
    df_driver_summary_base,
    "growth_log_capital_per_worker",
    "growth_log_hc",
)
print(
    f"Overall Pearson correlation between capital deepening and human capital growth: r={overall_r:.4f}, p={overall_p:.4g}, n={overall_n}"
)

rows = []
for grp in ["Q1_lowest", "Q2", "Q3", "Q4_highest"]:
    df_grp = df_driver_summary_base[
        df_driver_summary_base["income_quartile"] == grp
    ].copy()
    grp_r, grp_p, grp_n = _safe_corr(
        df_grp, "growth_log_capital_per_worker", "growth_log_hc"
    )
    rows.append(
        {
            "income_quartile": grp,
            "pearson_r": grp_r,
            "p_value": grp_p,
            "n_obs": grp_n,
        }
    )

df_group_corr = pd.DataFrame(rows)
print(
    "Pearson correlation between capital deepening and human capital growth by income group:"
)
print(df_group_corr)

# Compact interpretation focused on variability and composition
cap_std = df_driver_summary_base["growth_log_capital_per_worker"].std()
hc_std = df_driver_summary_base["growth_log_hc"].std()
cap_mean = df_driver_summary_base["growth_log_capital_per_worker"].mean()
hc_mean = df_driver_summary_base["growth_log_hc"].mean()

print("Conclusion:")
if pd.notna(overall_r) and abs(overall_r) < 0.2:
    print(
        "The two drivers are only weakly correlated overall, so their separate coefficients are unlikely to be driven mainly by simple collinearity."
    )
else:
    print(
        "The two drivers are meaningfully correlated overall, so coefficient comparisons should be interpreted with some caution."
    )

if pd.notna(hc_std) and pd.notna(cap_std) and hc_std < cap_std:
    print(
        "Human-capital growth varies less than capital deepening in this sample, which can make its coefficient look more stable or more precisely estimated even if the underlying productivity effect is not fundamentally larger."
    )
else:
    print(
        "Human-capital growth is not obviously lower-variance than capital deepening, so variance alone does not explain the coefficient pattern."
    )

print(
    "Because the estimates differ across income groups and the driver correlation is not uniform, the stronger human-capital coefficient may partly reflect measurement structure, subgroup composition, or limited within-group variation rather than a universally stronger causal effect on productivity growth."
)


## Task 4: Explore whether changes in capital deepening or human capital accumulation tend to precede productivity growth within countries using Granger causality tests on differenced series.

**Acceptance Criteria:** For a subset of countries with long, high-quality time series, Granger causality test results indicating whether past capital deepening or human capital growth help predict future TFP growth (and vice versa), with clear caveats about interpretation and sample limitations.

_Mode: exploratory_

### 4.1: Determine whether capital deepening or human capital growth Granger-cause TFP growth in a few representative countries with long, relatively complete series.
_Output: exploration_

In [ ]:
import numpy as np
import pandas as pd
from statsmodels.tsa.stattools import grangercausalitytests

# Build a clean exploration panel from the existing core data
_df_granger_base = df_pwt110_panel_core[
    [
        "countrycode",
        "country",
        "year",
        "growth_log_tfp_level",
        "growth_log_capital_per_worker",
        "growth_log_hc",
        "i_outlier",
        "i_irr",
    ]
].copy()

for col in [
    "year",
    "growth_log_tfp_level",
    "growth_log_capital_per_worker",
    "growth_log_hc",
]:
    _df_granger_base[col] = pd.to_numeric(_df_granger_base[col], errors="coerce")

# Flag obvious outliers using the existing quality flags
_df_granger_base["outlier_flag"] = (
    _df_granger_base["i_outlier"].astype("object").eq("Outlier")
    | _df_granger_base["i_irr"].astype("object").eq("Outlier")
).astype(int)

# Identify countries with long consecutive usable runs for each driver-target pair
_df_run_stats = []
for driver_col, driver_label in [
    ("growth_log_capital_per_worker", "capital_deepening"),
    ("growth_log_hc", "human_capital"),
]:
    _df_tmp = (
        _df_granger_base[
            [
                "countrycode",
                "country",
                "year",
                "growth_log_tfp_level",
                driver_col,
                "outlier_flag",
            ]
        ]
        .dropna(subset=["growth_log_tfp_level", driver_col])
        .copy()
    )
    _df_tmp = _df_tmp.sort_values(["countrycode", "year"]).reset_index(drop=True)
    _df_tmp["year_gap"] = _df_tmp.groupby("countrycode")["year"].diff()
    _df_tmp["run_id"] = (
        (_df_tmp["year_gap"].isna() | (_df_tmp["year_gap"] != 1))
        .groupby(_df_tmp["countrycode"])
        .cumsum()
    )
    _df_run = (
        _df_tmp.groupby(["countrycode", "country", "run_id"], dropna=False)
        .agg(
            run_len=("year", "size"),
            outlier_share=("outlier_flag", "mean"),
        )
        .reset_index()
    )
    _df_best = (
        _df_run.groupby(["countrycode", "country"], dropna=False)
        .agg(
            best_run_len=("run_len", "max"),
            best_outlier_share=("outlier_share", "min"),
        )
        .reset_index()
    )
    _df_best["driver"] = driver_label
    _df_run_stats.append(_df_best)

_df_country_runs = pd.concat(_df_run_stats, ignore_index=True).reset_index(drop=True)

# Select countries with at least about 25 usable observations and low outlier incidence
_df_candidate_countries = (
    _df_country_runs.query("best_run_len >= 25 and best_outlier_share <= 0.15")
    .sort_values(
        ["driver", "best_run_len", "best_outlier_share", "countrycode"],
        ascending=[True, False, True, True],
    )
    .reset_index(drop=True)
)

# Run Granger tests country by country for lags 1-3
_df_granger_rows = []
for driver_col, driver_label in [
    ("growth_log_capital_per_worker", "capital_deepening"),
    ("growth_log_hc", "human_capital"),
]:
    _df_candidates_driver = _df_candidate_countries[
        _df_candidate_countries["driver"] == driver_label
    ].copy()
    if _df_candidates_driver.empty:
        _df_candidates_driver = (
            _df_country_runs[_df_country_runs["driver"] == driver_label]
            .sort_values(
                ["best_run_len", "best_outlier_share", "countrycode"],
                ascending=[False, True, True],
            )
            .head(10)
            .copy()
        )

    for _, _row in _df_candidates_driver.iterrows():
        countrycode_val = _row["countrycode"]
        country_name_val = _row["country"]
        _df_country = _df_granger_base[
            _df_granger_base["countrycode"] == countrycode_val
        ].copy()
        _df_country = (
            _df_country[["year", "growth_log_tfp_level", driver_col, "outlier_flag"]]
            .dropna(subset=["growth_log_tfp_level", driver_col])
            .copy()
        )
        _df_country = _df_country.sort_values("year").reset_index(drop=True)

        # Need at least 10 usable observations to make lag-3 tests meaningful
        if len(_df_country) < 10:
            continue

        # Prepare directional test datasets
        _df_forward = _df_country[["growth_log_tfp_level", driver_col]].copy()
        _df_reverse = _df_country[[driver_col, "growth_log_tfp_level"]].copy()

        forward_pvals = {1: np.nan, 2: np.nan, 3: np.nan}
        reverse_pvals = {1: np.nan, 2: np.nan, 3: np.nan}

        try:
            _forward_results = grangercausalitytests(
                _df_forward, maxlag=3, verbose=False
            )
            for lag in [1, 2, 3]:
                forward_pvals[lag] = float(_forward_results[lag][0]["ssr_ftest"][1])
        except Exception:
            pass

        try:
            _reverse_results = grangercausalitytests(
                _df_reverse, maxlag=3, verbose=False
            )
            for lag in [1, 2, 3]:
                reverse_pvals[lag] = float(_reverse_results[lag][0]["ssr_ftest"][1])
        except Exception:
            pass

        _df_granger_rows.append(
            {
                "country": country_name_val,
                "countrycode": countrycode_val,
                "driver": driver_label,
                "direction": f"{driver_label} -> productivity",
                "n_obs": len(_df_country),
                "lag1_p": forward_pvals[1],
                "lag2_p": forward_pvals[2],
                "lag3_p": forward_pvals[3],
                "any_lag_sig_5pct": bool(
                    any(pd.notna(v) and v < 0.05 for v in forward_pvals.values())
                ),
                "reverse_lag1_p": reverse_pvals[1],
                "reverse_lag2_p": reverse_pvals[2],
                "reverse_lag3_p": reverse_pvals[3],
                "reverse_any_lag_sig_5pct": bool(
                    any(pd.notna(v) and v < 0.05 for v in reverse_pvals.values())
                ),
                "best_run_len": int(_row["best_run_len"]),
                "best_outlier_share": float(_row["best_outlier_share"]),
            }
        )

_df_granger_results = pd.DataFrame(_df_granger_rows)

if _df_granger_results.empty:
    print("No countries met the minimum usable length for Granger-style testing.")
else:
    # Compact printed table
    df_granger_summary = _df_granger_results[
        [
            "country",
            "countrycode",
            "driver",
            "direction",
            "n_obs",
            "best_run_len",
            "best_outlier_share",
            "lag1_p",
            "lag2_p",
            "lag3_p",
            "any_lag_sig_5pct",
            "reverse_lag1_p",
            "reverse_lag2_p",
            "reverse_lag3_p",
            "reverse_any_lag_sig_5pct",
        ]
    ].copy()

    # Friendly formatting for p-values
    for col in [
        "lag1_p",
        "lag2_p",
        "lag3_p",
        "reverse_lag1_p",
        "reverse_lag2_p",
        "reverse_lag3_p",
    ]:
        df_granger_summary[col] = df_granger_summary[col].map(
            lambda v: np.nan if pd.isna(v) else round(float(v), 4)
        )
    df_granger_summary["best_outlier_share"] = df_granger_summary[
        "best_outlier_share"
    ].map(lambda v: round(float(v), 3))

    print("Country-level Granger-style predictive precedence results (lags 1 to 3):")
    print(
        df_granger_summary.sort_values(
            ["driver", "any_lag_sig_5pct", "best_run_len", "countrycode"],
            ascending=[True, False, False, True],
        ).reset_index(drop=True)
    )

    # Cross-country summary: which driver more often predicts productivity growth?
    _df_forward_only = _df_granger_results.copy()
    _df_forward_only["sig_count"] = _df_forward_only[
        ["lag1_p", "lag2_p", "lag3_p"]
    ].apply(lambda r: int(any(pd.notna(v) and v < 0.05 for v in r.values)), axis=1)
    _df_forward_only["reverse_sig_count"] = _df_forward_only[
        ["reverse_lag1_p", "reverse_lag2_p", "reverse_lag3_p"]
    ].apply(lambda r: int(any(pd.notna(v) and v < 0.05 for v in r.values)), axis=1)

    df_driver_precedence = (
        _df_forward_only.groupby("driver", dropna=False)
        .agg(
            countries_tested=("countrycode", "nunique"),
            countries_forward_sig=("sig_count", "sum"),
            countries_reverse_sig=("reverse_sig_count", "sum"),
            avg_best_run_len=("best_run_len", "mean"),
            avg_outlier_share=("best_outlier_share", "mean"),
        )
        .reset_index()
    )
    df_driver_precedence["forward_sig_rate"] = (
        df_driver_precedence["countries_forward_sig"]
        / df_driver_precedence["countries_tested"]
    )
    df_driver_precedence["reverse_sig_rate"] = (
        df_driver_precedence["countries_reverse_sig"]
        / df_driver_precedence["countries_tested"]
    )

    print("Cross-country summary of predictive precedence:")
    print(df_driver_precedence)

    cap_sig = (
        int(
            df_driver_precedence.loc[
                df_driver_precedence["driver"] == "capital_deepening",
                "countries_forward_sig",
            ].iloc[0]
        )
        if (df_driver_precedence["driver"] == "capital_deepening").any()
        else 0
    )
    hc_sig = (
        int(
            df_driver_precedence.loc[
                df_driver_precedence["driver"] == "human_capital",
                "countries_forward_sig",
            ].iloc[0]
        )
        if (df_driver_precedence["driver"] == "human_capital").any()
        else 0
    )

    if cap_sig > hc_sig:
        precedence_statement = "Capital deepening more often shows predictive precedence for productivity growth in these country-level Granger checks."
    elif hc_sig > cap_sig:
        precedence_statement = "Human capital growth more often shows predictive precedence for productivity growth in these country-level Granger checks."
    else:
        precedence_statement = "Capital deepening and human capital growth show similar frequencies of predictive precedence in these country-level Granger checks."

    print(precedence_statement)
    print(
        "Note: these are predictive precedence tests, not structural causality claims."
    )


## Task 5: Integrate descriptive evidence, panel regressions, heterogeneity analysis, robustness checks, and Granger-causality insights to directly answer whether capital deepening or human capital accumulation is the primary driver of productivity growth across countries.

**Acceptance Criteria:** A clear, evidence-backed conclusion stating which factor appears more influential overall and under what conditions, with quantitative references to key coefficients and patterns, and a brief discussion of caveats and outliers.

### 5.1: Summarize and compare the main regression-based evidence on the relative strength of capital deepening and human capital growth as drivers of productivity growth.
_Output: print_

In [ ]:
import numpy as np
import pandas as pd

# Use existing objects; do not recreate upstream data/models
required_objects = [
    "df_pwt110_panel_reg_fe",
    "model_fe_base",
    "model_income_interactions",
    "df_income_effects",
]
for obj_name in required_objects:
    if obj_name not in globals():
        raise ValueError(f"{obj_name} is required but not found in the namespace.")

# Clean estimation sample aligned to the joint FE model

df_effect_base = df_pwt110_panel_reg_fe[
    [
        "countrycode",
        "country",
        "year",
        "growth_log_tfp_level",
        "growth_log_capital_per_worker",
        "growth_log_hc",
        "income_quartile",
    ]
].copy()

for col in [
    "growth_log_tfp_level",
    "growth_log_capital_per_worker",
    "growth_log_hc",
    "year",
]:
    df_effect_base[col] = pd.to_numeric(df_effect_base[col], errors="coerce")

df_effect_base = df_effect_base.dropna(
    subset=[
        "growth_log_tfp_level",
        "growth_log_capital_per_worker",
        "growth_log_hc",
        "income_quartile",
        "countrycode",
    ]
).copy()

# Overall standardized effects from the joint FE model coefficients
cap_sd = float(df_effect_base["growth_log_capital_per_worker"].std())
hc_sd = float(df_effect_base["growth_log_hc"].std())

cap_coef = float(model_fe_base.params["growth_log_capital_per_worker"])
hc_coef = float(model_fe_base.params["growth_log_hc"])

cap_ci = (
    model_fe_base.conf_int().loc["growth_log_capital_per_worker"].astype(float).tolist()
)
hc_ci = model_fe_base.conf_int().loc["growth_log_hc"].astype(float).tolist()

rows_overall = [
    {
        "driver": "capital_deepening",
        "coef": cap_coef,
        "ci_low": float(cap_ci[0]),
        "ci_high": float(cap_ci[1]),
        "sd_regressor": cap_sd,
        "std_effect": cap_coef * cap_sd,
        "std_ci_low": float(cap_ci[0]) * cap_sd,
        "std_ci_high": float(cap_ci[1]) * cap_sd,
        "p_value": float(model_fe_base.pvalues["growth_log_capital_per_worker"]),
        "sig_5pct": bool(model_fe_base.pvalues["growth_log_capital_per_worker"] < 0.05),
        "scope": "overall",
        "income_quartile": "all",
    },
    {
        "driver": "human_capital",
        "coef": hc_coef,
        "ci_low": float(hc_ci[0]),
        "ci_high": float(hc_ci[1]),
        "sd_regressor": hc_sd,
        "std_effect": hc_coef * hc_sd,
        "std_ci_low": float(hc_ci[0]) * hc_sd,
        "std_ci_high": float(hc_ci[1]) * hc_sd,
        "p_value": float(model_fe_base.pvalues["growth_log_hc"]),
        "sig_5pct": bool(model_fe_base.pvalues["growth_log_hc"] < 0.05),
        "scope": "overall",
        "income_quartile": "all",
    },
]

# Reconstruct group-specific coefficients from the income interaction model
cat_series = pd.Categorical(df_effect_base["income_quartile"])
base_group = cat_series.categories[0]
all_groups = list(cat_series.categories)

cov_mat = model_income_interactions.cov_params()
base_cap_term = "growth_log_capital_per_worker"
base_hc_term = "growth_log_hc"

cap_base_coef = float(model_income_interactions.params.get(base_cap_term, 0.0))
hc_base_coef = float(model_income_interactions.params.get(base_hc_term, 0.0))
cap_base_var = (
    float(cov_mat.loc[base_cap_term, base_cap_term])
    if base_cap_term in cov_mat.index
    else np.nan
)
hc_base_var = (
    float(cov_mat.loc[base_hc_term, base_hc_term])
    if base_hc_term in cov_mat.index
    else np.nan
)

for grp in all_groups:
    if grp == base_group:
        cap_term = base_cap_term
        hc_term = base_hc_term
        cap_group_coef = cap_base_coef
        hc_group_coef = hc_base_coef
        cap_group_se = (
            float(np.sqrt(max(cap_base_var, 0.0))) if pd.notna(cap_base_var) else np.nan
        )
        hc_group_se = (
            float(np.sqrt(max(hc_base_var, 0.0))) if pd.notna(hc_base_var) else np.nan
        )
        cap_group_p = float(model_income_interactions.pvalues.get(cap_term, np.nan))
        hc_group_p = float(model_income_interactions.pvalues.get(hc_term, np.nan))
    else:
        cap_term = f"growth_log_capital_per_worker:C(income_quartile)[T.{grp}]"
        hc_term = f"growth_log_hc:C(income_quartile)[T.{grp}]"
        cap_int = float(model_income_interactions.params.get(cap_term, 0.0))
        hc_int = float(model_income_interactions.params.get(hc_term, 0.0))
        cap_group_coef = cap_base_coef + cap_int
        hc_group_coef = hc_base_coef + hc_int

        cap_var = np.nan
        hc_var = np.nan
        if cap_term in cov_mat.index:
            cap_var = (
                cov_mat.loc[base_cap_term, base_cap_term]
                + cov_mat.loc[cap_term, cap_term]
                + 2.0 * cov_mat.loc[base_cap_term, cap_term]
            )
        if hc_term in cov_mat.index:
            hc_var = (
                cov_mat.loc[base_hc_term, base_hc_term]
                + cov_mat.loc[hc_term, hc_term]
                + 2.0 * cov_mat.loc[base_hc_term, hc_term]
            )
        cap_group_se = (
            float(np.sqrt(max(cap_var, 0.0))) if pd.notna(cap_var) else np.nan
        )
        hc_group_se = float(np.sqrt(max(hc_var, 0.0))) if pd.notna(hc_var) else np.nan
        cap_group_p = float(model_income_interactions.pvalues.get(cap_term, np.nan))
        hc_group_p = float(model_income_interactions.pvalues.get(hc_term, np.nan))

    rows_overall.append(
        {
            "driver": "capital_deepening",
            "coef": cap_group_coef,
            "ci_low": np.nan,
            "ci_high": np.nan,
            "sd_regressor": cap_sd,
            "std_effect": cap_group_coef * cap_sd,
            "std_ci_low": np.nan,
            "std_ci_high": np.nan,
            "p_value": cap_group_p,
            "sig_5pct": bool(pd.notna(cap_group_p) and cap_group_p < 0.05),
            "scope": "income_group",
            "income_quartile": str(grp),
        }
    )
    rows_overall.append(
        {
            "driver": "human_capital",
            "coef": hc_group_coef,
            "ci_low": np.nan,
            "ci_high": np.nan,
            "sd_regressor": hc_sd,
            "std_effect": hc_group_coef * hc_sd,
            "std_ci_low": np.nan,
            "std_ci_high": np.nan,
            "p_value": hc_group_p,
            "sig_5pct": bool(pd.notna(hc_group_p) and hc_group_p < 0.05),
            "scope": "income_group",
            "income_quartile": str(grp),
        }
    )

# Final workspace dataframe

df_driver_effects_by_income = pd.DataFrame(rows_overall)

df_driver_effects_by_income = df_driver_effects_by_income[
    [
        "scope",
        "income_quartile",
        "driver",
        "coef",
        "ci_low",
        "ci_high",
        "sd_regressor",
        "std_effect",
        "std_ci_low",
        "std_ci_high",
        "p_value",
        "sig_5pct",
    ]
].copy()

# Create compact print tables

df_overall_std = df_driver_effects_by_income[
    df_driver_effects_by_income["scope"] == "overall"
].copy()
df_income_effects_summary = df_driver_effects_by_income[
    df_driver_effects_by_income["scope"] == "income_group"
].copy()

print("Overall standardized effects from the joint fixed-effects model:")
print(
    df_overall_std[
        [
            "driver",
            "coef",
            "ci_low",
            "ci_high",
            "sd_regressor",
            "std_effect",
            "std_ci_low",
            "std_ci_high",
            "p_value",
            "sig_5pct",
        ]
    ]
)
print("Income-group-specific reconstructed effects:")
print(
    df_income_effects_summary[
        ["income_quartile", "driver", "coef", "p_value", "sig_5pct", "std_effect"]
    ]
)

add_to_workspace(df_driver_effects_by_income)


### 5.2: Identify countries where productivity growth is high or low despite weak measured growth in both capital per worker and human capital, signaling other factors or data issues.
_Output: print_

In [ ]:
import numpy as np
import pandas as pd

# Use existing objects; do not recreate upstream data/models
required_objects = [
    "df_pwt110_panel_reg_fe",
    "model_fe_base",
    "model_income_interactions",
    "df_income_effects",
]
for obj_name in required_objects:
    if obj_name not in globals():
        raise ValueError(f"{obj_name} is required but not found in the namespace.")

# Clean estimation sample aligned to the joint FE model

df_effect_base = df_pwt110_panel_reg_fe[
    [
        "countrycode",
        "country",
        "year",
        "growth_log_tfp_level",
        "growth_log_capital_per_worker",
        "growth_log_hc",
        "income_quartile",
    ]
].copy()

for col in [
    "growth_log_tfp_level",
    "growth_log_capital_per_worker",
    "growth_log_hc",
    "year",
]:
    df_effect_base[col] = pd.to_numeric(df_effect_base[col], errors="coerce")

df_effect_base = df_effect_base.dropna(
    subset=[
        "growth_log_tfp_level",
        "growth_log_capital_per_worker",
        "growth_log_hc",
        "income_quartile",
        "countrycode",
    ]
).copy()

# Overall standardized effects from the joint FE model coefficients
cap_sd = float(df_effect_base["growth_log_capital_per_worker"].std())
hc_sd = float(df_effect_base["growth_log_hc"].std())

cap_coef = float(model_fe_base.params["growth_log_capital_per_worker"])
hc_coef = float(model_fe_base.params["growth_log_hc"])

cap_ci = (
    model_fe_base.conf_int().loc["growth_log_capital_per_worker"].astype(float).tolist()
)
hc_ci = model_fe_base.conf_int().loc["growth_log_hc"].astype(float).tolist()

rows_overall = [
    {
        "driver": "capital_deepening",
        "coef": cap_coef,
        "ci_low": float(cap_ci[0]),
        "ci_high": float(cap_ci[1]),
        "sd_regressor": cap_sd,
        "std_effect": cap_coef * cap_sd,
        "std_ci_low": float(cap_ci[0]) * cap_sd,
        "std_ci_high": float(cap_ci[1]) * cap_sd,
        "p_value": float(model_fe_base.pvalues["growth_log_capital_per_worker"]),
        "sig_5pct": bool(model_fe_base.pvalues["growth_log_capital_per_worker"] < 0.05),
        "scope": "overall",
        "income_quartile": "all",
    },
    {
        "driver": "human_capital",
        "coef": hc_coef,
        "ci_low": float(hc_ci[0]),
        "ci_high": float(hc_ci[1]),
        "sd_regressor": hc_sd,
        "std_effect": hc_coef * hc_sd,
        "std_ci_low": float(hc_ci[0]) * hc_sd,
        "std_ci_high": float(hc_ci[1]) * hc_sd,
        "p_value": float(model_fe_base.pvalues["growth_log_hc"]),
        "sig_5pct": bool(model_fe_base.pvalues["growth_log_hc"] < 0.05),
        "scope": "overall",
        "income_quartile": "all",
    },
]

# Reconstruct group-specific coefficients from the income interaction model
cat_series = pd.Categorical(df_effect_base["income_quartile"])
base_group = cat_series.categories[0]
all_groups = list(cat_series.categories)

cov_mat = model_income_interactions.cov_params()
base_cap_term = "growth_log_capital_per_worker"
base_hc_term = "growth_log_hc"

cap_base_coef = float(model_income_interactions.params.get(base_cap_term, 0.0))
hc_base_coef = float(model_income_interactions.params.get(base_hc_term, 0.0))
cap_base_var = (
    float(cov_mat.loc[base_cap_term, base_cap_term])
    if base_cap_term in cov_mat.index
    else np.nan
)
hc_base_var = (
    float(cov_mat.loc[base_hc_term, base_hc_term])
    if base_hc_term in cov_mat.index
    else np.nan
)

for grp in all_groups:
    if grp == base_group:
        cap_term = base_cap_term
        hc_term = base_hc_term
        cap_group_coef = cap_base_coef
        hc_group_coef = hc_base_coef
        cap_group_se = (
            float(np.sqrt(max(cap_base_var, 0.0))) if pd.notna(cap_base_var) else np.nan
        )
        hc_group_se = (
            float(np.sqrt(max(hc_base_var, 0.0))) if pd.notna(hc_base_var) else np.nan
        )
        cap_group_p = float(model_income_interactions.pvalues.get(cap_term, np.nan))
        hc_group_p = float(model_income_interactions.pvalues.get(hc_term, np.nan))
    else:
        cap_term = f"growth_log_capital_per_worker:C(income_quartile)[T.{grp}]"
        hc_term = f"growth_log_hc:C(income_quartile)[T.{grp}]"
        cap_int = float(model_income_interactions.params.get(cap_term, 0.0))
        hc_int = float(model_income_interactions.params.get(hc_term, 0.0))
        cap_group_coef = cap_base_coef + cap_int
        hc_group_coef = hc_base_coef + hc_int

        cap_var = np.nan
        hc_var = np.nan
        if cap_term in cov_mat.index:
            cap_var = (
                cov_mat.loc[base_cap_term, base_cap_term]
                + cov_mat.loc[cap_term, cap_term]
                + 2.0 * cov_mat.loc[base_cap_term, cap_term]
            )
        if hc_term in cov_mat.index:
            hc_var = (
                cov_mat.loc[base_hc_term, base_hc_term]
                + cov_mat.loc[hc_term, hc_term]
                + 2.0 * cov_mat.loc[base_hc_term, hc_term]
            )
        cap_group_se = (
            float(np.sqrt(max(cap_var, 0.0))) if pd.notna(cap_var) else np.nan
        )
        hc_group_se = float(np.sqrt(max(hc_var, 0.0))) if pd.notna(hc_var) else np.nan
        cap_group_p = float(model_income_interactions.pvalues.get(cap_term, np.nan))
        hc_group_p = float(model_income_interactions.pvalues.get(hc_term, np.nan))

    rows_overall.append(
        {
            "driver": "capital_deepening",
            "coef": cap_group_coef,
            "ci_low": np.nan,
            "ci_high": np.nan,
            "sd_regressor": cap_sd,
            "std_effect": cap_group_coef * cap_sd,
            "std_ci_low": np.nan,
            "std_ci_high": np.nan,
            "p_value": cap_group_p,
            "sig_5pct": bool(pd.notna(cap_group_p) and cap_group_p < 0.05),
            "scope": "income_group",
            "income_quartile": str(grp),
        }
    )
    rows_overall.append(
        {
            "driver": "human_capital",
            "coef": hc_group_coef,
            "ci_low": np.nan,
            "ci_high": np.nan,
            "sd_regressor": hc_sd,
            "std_effect": hc_group_coef * hc_sd,
            "std_ci_low": np.nan,
            "std_ci_high": np.nan,
            "p_value": hc_group_p,
            "sig_5pct": bool(pd.notna(hc_group_p) and hc_group_p < 0.05),
            "scope": "income_group",
            "income_quartile": str(grp),
        }
    )

# Final workspace dataframe

df_driver_effects_by_income = pd.DataFrame(rows_overall)

df_driver_effects_by_income = df_driver_effects_by_income[
    [
        "scope",
        "income_quartile",
        "driver",
        "coef",
        "ci_low",
        "ci_high",
        "sd_regressor",
        "std_effect",
        "std_ci_low",
        "std_ci_high",
        "p_value",
        "sig_5pct",
    ]
].copy()

# Create compact print tables

df_overall_std = df_driver_effects_by_income[
    df_driver_effects_by_income["scope"] == "overall"
].copy()
df_income_effects_summary = df_driver_effects_by_income[
    df_driver_effects_by_income["scope"] == "income_group"
].copy()

print("Overall standardized effects from the joint fixed-effects model:")
print(
    df_overall_std[
        [
            "driver",
            "coef",
            "ci_low",
            "ci_high",
            "sd_regressor",
            "std_effect",
            "std_ci_low",
            "std_ci_high",
            "p_value",
            "sig_5pct",
        ]
    ]
)
print("Income-group-specific reconstructed effects:")
print(
    df_income_effects_summary[
        ["income_quartile", "driver", "coef", "p_value", "sig_5pct", "std_effect"]
    ]
)

add_to_workspace(df_driver_effects_by_income)


### 5.3: Produce a simple visualization that communicates how the relative importance of capital deepening versus human capital growth differs across income groups.
_Output: figure_

_No successful execution recorded for this subtask._

### 5.4: Cross-check key findings for internal consistency and directly answer whether capital deepening or human capital accumulation is the main driver of productivity growth across countries, noting heterogeneity and limitations.
_Output: print_

_No successful execution recorded for this subtask._